# Day 3a. Middleware: the general machine

Yesterday the loop was a black box: messages in, answer out. Today you reach
**inside** it.

A middleware is a class with **hooks**, six places where your code runs while
the agent is working. In this notebook: watch the hooks fire, use a shipped one
that keeps the context window from filling, and write your own.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

### Restore yesterday's world

Every day-3 notebook starts from the same booking agent you built on day 2.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

HOTELS = {"Hotel Anders":  {"free": True,  "eur": 92},
          "Pension Krumm": {"free": True,  "eur": 61},
          "Grand Mitte":   {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = ("You are the GIU booking assistant. "
                "Check availability before booking. Prices in EUR.")
print("world restored")

## 1 · Watch the hooks fire

The agent loop goes round and round: the model proposes, tools run, results come
back, the model proposes again. It stops when the model answers instead of
asking for a tool.

There are six places where your code can sit:

| hook | when it fires | what you'd do there |
|---|---|---|
| `before_agent` | once, at the start | load what the whole run needs |
| `before_model` | every lap | edit the list the model is about to read |
| `wrap_model_call` | around the model call | swap the model or the prompt, retry |
| `after_model` | every lap, after it answers | check what it said, before tools run |
| `wrap_tool_call` | once per tool call | allow, rewrite, refuse, or **pause** |
| `after_agent` | once, at the end | log, meter, audit |

Two families, and the difference matters:

- `before_*` and `after_*` **look at the state** and may return a change.
- `wrap_*` hooks stand **in the doorway**. Nothing reaches the model or a tool
  except by you calling `handler(...)` yourself. Which means you can also not
  call it.

Run the next cell and watch the order. Ask yourself, before you run it: how many
times will `before_model` print?

In [ ]:
from langchain.agents.middleware import AgentMiddleware

class TraceMiddleware(AgentMiddleware):
    """Prints every hook as it fires. Pure observation, changes nothing."""

    def before_agent(self, state, runtime):
        print("┌─ before_agent · run starts")

    def before_model(self, state, runtime):
        print(f"│  before_model · model will see {len(state['messages'])} messages")

    def after_model(self, state, runtime):
        last = state["messages"][-1]
        calls = [c["name"] for c in getattr(last, "tool_calls", [])]
        print(f"│  after_model  · wants tools: {calls or 'none, it answered'}")

    def wrap_tool_call(self, request, handler):
        print(f"│  wrap_tool_call · {request.tool_call['name']}({request.tool_call['args']})")
        return handler(request)          # ← let it through

    def after_agent(self, state, runtime):
        print("└─ after_agent · run ends")

agent = create_agent(model=llm,
    tools=[check_availability, list_hotels],
    system_prompt=INSTRUCTIONS,
    middleware=[TraceMiddleware()])      # ← the new argument

out = agent.invoke({"messages": [HumanMessage("Cheapest room in Berlin?")]})
print("\nANSWER:", out["messages"][-1].content[:200])

Read that output before moving on. You just saw the loop from the inside:
`before_model` fires once per iteration, and the run only ends when the model
answers instead of calling a tool.

> **The agent code never changed.** You attached behavior; you did not rewrite
> the loop. That is the whole idea.

## 2 · A shipped one: the window that never fills

Day 2's problem: long conversations fill the context window, and quality drops.
`SummarizationMiddleware` fixes it at `before_model`, when the transcript
crosses your trigger, the oldest turns are replaced by a model-written summary.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

chatty = create_agent(model=llm,
    tools=[check_availability, list_hotels],
    system_prompt=INSTRUCTIONS,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 8),     # summarize once the list passes 8
            keep=("messages", 4),        # always keep the last 4 untouched
        ),
        TraceMiddleware(),               # so we can watch the count drop
    ])

msgs = []
for turn in ["Hi, I'm Amr.",
             "What's the cheapest hotel?",
             "Is Grand Mitte free?",
             "What about Hotel Anders?",
             "Remind me: what is my name, and which hotel was cheapest?"]:
    msgs = msgs + [HumanMessage(turn)]
    out = agent_out = chatty.invoke({"messages": msgs})
    msgs = out["messages"]
    print(f"── after '{turn[:32]}' → list is {len(msgs)} messages\n")

print("FINAL:", msgs[-1].content[:250])

Watch the `before_model` counts. They grow as the chat gets longer, and on the
turn that crosses the trigger you see the count **drop mid-turn**: the summary
replaced the old messages. The agent still knows your name, because the summary
carried it forward.

Triggers can be `("messages", n)`, `("tokens", n)` or `("fraction", 0.8)` of the
window. That is the knob you tune per application.

## 3 · Your turn, a budget you enforce

An agent stuck in a loop calls tools forever, and you pay for every call. Write
a **ToolBudgetMiddleware** that allows at most `max_calls` tool calls per run
and refuses politely after that.

Where: `wrap_tool_call`, it is the only hook that can decline. To refuse,
return a `ToolMessage` **instead of** calling `handler(request)`:

```python
return ToolMessage(content="...", tool_call_id=request.tool_call["id"])
```

The agent reads that message like any tool result and reacts to it.

In [ ]:
from langchain.messages import ToolMessage

class ToolBudgetMiddleware(AgentMiddleware):
    """Allow at most max_calls tool calls per run."""

    def __init__(self, max_calls: int = 2):
        super().__init__()
        self.max_calls = max_calls
        self.used = 0

    def before_agent(self, state, runtime):
        self.used = 0                     # fresh budget every run

    def wrap_tool_call(self, request, handler):
        self.used += 1
        # YOUR CODE: when self.used > self.max_calls, return a ToolMessage
        # saying the tool is unavailable (see 'It can also refuse' below).
        # For now it just lets everything through, so the cell runs:
        return handler(request)

budgeted = create_agent(model=llm,
    tools=[check_availability, list_hotels],
    system_prompt=INSTRUCTIONS,
    middleware=[ToolBudgetMiddleware(max_calls=2), TraceMiddleware()])

budgeted.invoke({"messages": [HumanMessage(
    "Check every single hotel one by one, then tell me the cheapest.")]})["messages"][-1].content

**Then discuss:** your middleware counts calls on `self`. What breaks if two
users hit the same agent object at the same time? (Hint: where *should* per-run
state live? Look at what `before_agent` receives.)

**Bonus**, the shipped versions: `ToolCallLimitMiddleware` and
`ModelCallLimitMiddleware`. Same idea, hardened. Same pattern as every day this
week: build one to understand it, then take the shipped one.

In [ ]:
# your experiments here


---
## Wrap

Six hooks around the loop · `middleware=[...]` on the same `create_agent` call ·
observe with `before_/after_`, intercept with `wrap_`. Summarization keeps the
window honest; your own middleware enforces a budget.

**Next (3b):** one hook can also **stop** the loop, and wait for a human.